In [18]:
import pystac
import os
import xarray as xr
from netCDF4 import Dataset
import numpy as np
from datetime import datetime, timezone
from shapely.geometry import Polygon, mapping
import pandas as pd

test_dir = "/work/a3r/TFTEST"

dirs = [None, None, None, 'experiment_id', 'member_id', 'realm', 'cell_methods', 'frequency', 'chunk_freq']
fname = ["realm", "time_range", "variable_id"]
all_columns = ["activity_id", "institution_id", "source_id", "experiment_id",
                "frequency", "realm", "table_id",
                "member_id", "grid_label", "variable_id",
                "time_range", "chunk_freq","platform","dimensions",
                "cell_methods","standard_name","path"]
props_template = {c : '' for c in all_columns}

In [34]:
class MetadataSlow():
    def __init__(self, lats, lons, long_name):
        self.lats = lats
        self.lons = lons
        self.long_name = long_name

class MetadataSlowLoader():
    def __init__(self):
        self.dict = {}

    def _get_bbox_footprint(self, variable_id):
        bottom,top = self.dict[variable_id].lats
        left,right = self.dict[variable_id].lons

        bbox = [left, bottom, right, top]
        footprint = Polygon([
            [left, bottom],
            [left, top],
            [right, top],
            [right, bottom]
        ])

        return (bbox, mapping(footprint))
    
    def get(self, path, properties):
        var_id = properties["variable_id"]
        if var_id not in self.dict:
            ds = Dataset(path, memory=None)

            if properties["realm"] == "ocean":
                top = float(np.max(ds.variables["yh"]))
                bottom = float(np.min(ds.variables["yh"]))
                left = float(np.min(ds.variables["xh"]))
                right = float(np.max(ds.variables["xh"]))
            else:
                top = float(np.max(ds.variables["lat"]))
                bottom = float(np.min(ds.variables["lat"]))
                left = float(np.min(ds.variables["lon"]))
                right = float(np.max(ds.variables["lon"]))

            long_name = ds.variables[var_id].long_name
            self.dict[var_id] = MetadataSlow([bottom,top], [left,right], long_name)
        
        bbox,fp = self._get_bbox_footprint(var_id)
        return (bbox,fp,self.dict[var_id].long_name)

def get_metadata(path_to_file, dir_meta, file_meta, properties=props_template):
    d = dict(properties)
    filename = os.path.basename(path_to_file).split('.')
    dir_structure = os.path.dirname(path_to_file).split('/')[1:]
    for (i,x) in enumerate(dir_meta):
        if x is not None:
            d[x] = dir_structure[i]
    for (i,x) in enumerate(file_meta):
        if x is not None:
            d[x] = filename[i]
    starttime, endtime = d["time_range"].split('-')
    starttime = datetime.fromisoformat(starttime).replace(tzinfo=timezone.utc)
    endtime = datetime.fromisoformat(endtime).replace(tzinfo=timezone.utc)
    return (d, starttime, endtime)

def make_catalog(directory, dir_meta=dirs, f_meta=fname):
    catalog = pystac.Catalog(id="test-catalog", description="Test Catalog")
    
    item_dicts = {}
    slow_metadata = MetadataSlowLoader()

    files = [os.path.join(dirpath,f) for (dirpath, dirnames, filenames) in 
             os.walk(directory) for f in filenames]
    N_files = len(files)
    digits_files = int(np.ceil(np.log10(N_files)))
    IDs = [str(i).zfill(digits_files) for i in range(N_files)]

    for i,f in enumerate(files):
        print(f)
        metadata_d, stime, etime = get_metadata(f, dir_meta, f_meta)

        exp_id = metadata_d["experiment_id"]
        if exp_id not in item_dicts:
            item_dicts[exp_id] = {}
        exp_dict = item_dicts[exp_id]

        var_id = metadata_d["variable_id"]
        if var_id not in exp_dict:
            bbox, footprint, long_name = slow_metadata.get(f, metadata_d)
            metadata_d["standard_name"] = long_name

            exp_dict[var_id] = pystac.Item(id=IDs[i], geometry=footprint, 
                bbox=bbox, properties=dict(metadata_d), start_datetime=stime, 
                end_datetime=etime, datetime=None)
            
            del exp_dict[var_id].properties["time_range"]
            del exp_dict[var_id].properties["member_id"]

        else:
            item_start = datetime.fromisoformat(exp_dict[var_id].properties["start_datetime"])
            if stime < item_start:
                exp_dict[var_id].properties["start_datetime"] = stime.replace(tzinfo=None).isoformat()+'Z'

            item_end = datetime.fromisoformat(exp_dict[var_id].properties["end_datetime"])
            if etime > item_end:
                exp_dict[var_id].properties["end_datetime"] = etime.replace(tzinfo=None).isoformat()+'Z'

        asset = pystac.Asset(href=f, title="{}.{}".format(metadata_d["member_id"], metadata_d["time_range"]))

        exp_dict[var_id].add_asset(key="dataset", asset=asset)
    
    collections = []
    for (k,v) in item_dicts.items():
        collections.append(pystac.Collection(
                id=exp_id,
                description="NA",
                extent=pystac.collection.Extent(
                    pystac.collection.SpatialExtent([item.bbox for item in v.values()]),
                    pystac.collection.TemporalExtent([[
                        item.properties["start_datetime"], 
                        item.properties["end_datetime"]
                    ] for item in v.values()])
                )
            ))
        for item in v.values():
            collections[-1].add_item(item)
    
    for coll in collections:
        catalog.add_child(coll)

    return catalog


In [35]:
make_catalog(test_dir)

/work/a3r/TFTEST/SPEAR_c192_o1_Hist_AllForc_IC1921_K50/pp_ens_01/atmos_daily/ts/daily/10yr/atmos_daily.19210101-19301231.t_ref_max.nc
/work/a3r/TFTEST/SPEAR_c192_o1_Hist_AllForc_IC1921_K50/pp_ens_01/atmos_daily/ts/daily/10yr/atmos_daily.19310101-19401231.t_ref_max.nc


id: test-catalog
description: Test Catalog
id: SPEAR_c192_o1_Hist_AllForc_IC1921_K50
description: NA
id: 0
"bbox: [0.3125, -89.75, 359.6875, 89.75]"
activity_id:
institution_id:
source_id:
experiment_id: SPEAR_c192_o1_Hist_AllForc_IC1921_K50
frequency: daily


In [21]:
%timeit catalog2 = pd.read_csv("/home/Aria.Radick/Documents/spear-flp/catalog_blue.csv")

12 ms ± 426 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [19]:
datetime.fromisoformat("19310101").replace(tzinfo=timezone.utc)

datetime.datetime(1931, 1, 1, 0, 0, tzinfo=datetime.timezone.utc)

In [30]:
datetime.fromisoformat(items[0].properties["start_datetime"])

datetime.datetime(1931, 1, 1, 0, 0, tzinfo=datetime.timezone.utc)

In [4]:
catalog = pystac.Catalog.from_file("/home/a3r/Documents/stac_test/catalog.json")

In [5]:
items = list(catalog.get_items())

In [40]:
type(items[0].properties['end_datetime'])

str

In [43]:
datetime.fromisoformat(items[0].properties["start_datetime"]).replace(tzinfo=None).isoformat()+'Z'

'1931-01-01T00:00:00Z'

In [14]:
%%timeit
results = []
for x in catalog.get_items():
    if x.properties["variable_id"] == "snow":
        results.append(x)

9.29 ms ± 59.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
%timeit catalog2[catalog2["variable_id"] == "snow"]

348 μs ± 1.01 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
